# 📄 Document Extraction Engine — Kaggle Demo
Single-stage multimodal extraction with **Qwen3-VL-8B** (or Qwen2.5-VL), visual bounding boxes, and deterministic financial normalization.

In [ ]:
# 1. Clean workspace & clone target branch
import os, shutil

shutil.rmtree('/kaggle/working/DOC_Extractor_QUAPT', ignore_errors=True)

!git clone -b feature/prd-v1.3-qwen3-vl-multimodal https://github.com/dharicharan/DOC_Extractor_QUAPT.git /kaggle/working/DOC_Extractor_QUAPT

# 2. Install Astral UV for ultra-fast package management
!curl -Ls https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]

# 3. Install dependencies
%cd /kaggle/working/DOC_Extractor_QUAPT
!uv pip install --system -r requirements.txt
!uv pip install --system -e .
!uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio

print("✅ Setup and dependencies complete!")

In [ ]:
# (Optional) Warm up / Pre-download Qwen3-VL-8B with 4-bit quantization
# Fits comfortably in Kaggle's free T4 16GB VRAM
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

model_id = "Qwen/Qwen3-VL-8B-Instruct"

print("📥 Downloading processor...")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=1280*28*28)

print("📥 Downloading & quantizing model (4-bit)... please wait 1-2 minutes")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
print(f"✅ Model cached on {model.device}!")

In [ ]:
# Free any ports from previous runs and launch Gradio with public share link
!fuser -k 7860/tcp 2>/dev/null || true

%cd /kaggle/working/DOC_Extractor_QUAPT/src/doc_extractor
!python app.py